This notebook provides the code to generate the scientist-inventors based PPPs as well as the metadata corresponding to the PPPs.

## Packages

In [ ]:
# ! pip install pandas networkx matplotlib metaphone fuzzywuzzy   
! pip install python-Levenshtein

In [ ]:
import pandas as pd
import json, requests 
import networkx as nx
import gc
# import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from tqdm import tqdm
import gc
import time
import sys
import unicodedata
from metaphone import doublemetaphone
from fuzzywuzzy import fuzz
# from difflib import SequenceMatcher
import re
from math import radians, cos, sin, asin, sqrt
import datetime 
from datetime import date

# window = (365*5)


set up general parameters

In [ ]:
maxpatentpaperyeardiff = 2
datadir = './data/'

In [ ]:
pcspairs = pd.read_csv(datadir + 'pcself.tsv', sep='\t')
pcspairs['paperid'] = pcspairs['paperid'].astype(int)
pcspairs['paperid'] = pcspairs['paperid'].astype(str)
pcspairs['patent_id'] = pcspairs['patent_id'].astype(str)
pcspairs.rename(columns={'paperid':'work_id'}, inplace=True)
print(pcspairs.shape)


In [ ]:
pqrpapers = pd.read_csv(datadir + 'pqrs_authorid_workid.tsv', sep='\t')
pqrpatents = pd.read_csv(datadir + 'inventorid_patentid.tsv', sep='\t')
pqrs = pd.read_csv('pqrs_dataset.tsv', sep='\t')
pqrs = pqrs[['author_id', 'inventor_id', 'confidence']].drop_duplicates()
pqrs = pqrs[pqrs['confidence'] >= .95]
pqrpairs = pqrs.merge(pqrpapers, on='author_id', how='left')
del pqrpapers
pqrpairs = pqrpairs.merge(pqrpatents, on='inventor_id', how='left')
del pqrpatents
pqrpairs = pqrpairs[['work_id', 'patent_id']].drop_duplicates()
pqrpairs['patent_id'] = pqrpairs['patent_id'].str.replace('US-', '', regex=False)
print(pqrpairs)
gc.collect()

In [ ]:
potentialpairs = pd.concat([pcspairs, pqrpairs], ignore_index=True)
del pcspairs 
del pqrpairs
gc.collect()
potentialpairs.dropna(inplace=True)
potentialpairs

In [ ]:
paperyears = pd.read_csv(datadir + 'openalexidyear.tsv', sep='\t', on_bad_lines='skip', header=None) ## REPLACE WITH MAGDATE, MAKE IT EXACT 
paperyears.rename(columns={1: 'work_id', 0: 'paperyear'}, inplace=True) 
paperyears.dropna(inplace=True)
paperyears['paperyear'] = paperyears['paperyear'].astype('Int64')
paperyears['work_id'] = paperyears['work_id'].astype('Int64')
paperyears.head()                         

In [ ]:
patentdates = pd.read_csv(datadir + 'g_application.tsv.zip', sep='\t', compression='zip')
patentdates = patentdates[['patent_id', 'filing_date']].drop_duplicates()
# print(patentdates.head())
patentdates['patentyear'] = pd.to_datetime(patentdates['filing_date'], format='%Y-%m-%d', errors='coerce').dt.year.astype('Int64')
patentdates = patentdates[['patent_id',   'patentyear']].drop_duplicates()
# print(patentdates['patentyear'].value_counts())


In [ ]:
potentialpairs = potentialpairs.merge(paperyears, on='work_id', how='left')
del paperyears
potentialpairs = potentialpairs.merge(patentdates, on='patent_id', how='left')  
del patentdates
potentialpairs.dropna(inplace=True)
gc.collect()
potentialpairs

In [ ]:
potentialpairs['yearsdiff'] = abs(potentialpairs['patentyear'] - potentialpairs['paperyear'])
print(potentialpairs['yearsdiff'].value_counts())

# potentialpairs['inwindow'] = (potentialpairs['yearsdiff'])<3
# potentialpairs_window = potentialpairs[potentialpairs['inwindow'] == True]
# del potentialpairs
# gc.collect() 


shrink to pairs within {maxpatentpaperyeardiff}}

In [ ]:
potentialpairs = potentialpairs[potentialpairs['yearsdiff'] < maxpatentpaperyeardiff]
potentialpairs = potentialpairs[['work_id', 'patent_id']]
potentialpairs['work_id'] = potentialpairs['work_id'].astype(str)
potentialpairs['patent_id'] = potentialpairs['patent_id'].astype(str)
potentialpairs

In [ ]:
pqrspaperdataembed = pd.read_parquet(datadir + 'pqr_embeddedpapers.parquet')
pcsselfpaperdataembed = pd.read_parquet(datadir + 'pcsself_encoded_papers.parquet')
paperdataembed = pd.concat([pqrspaperdataembed, pcsselfpaperdataembed], ignore_index=True)
paperdataembed.columns 
del pqrspaperdataembed 
del pcsselfpaperdataembed 
gc.collect()
paperdataembed = paperdataembed.drop_duplicates(subset=['id'], keep='first')
paperdataembed = paperdataembed.rename(columns={'id': 'work_id'})
paperdataembed['work_id'] = paperdataembed['work_id'].astype(str)
paperdataembed = paperdataembed.rename(columns={'encoded_title': 'encoded_paper_title'})  
paperdataembed = paperdataembed.rename(columns={'encoded_abstract': 'encoded_paper_abstract'})  
print(paperdataembed.shape)
paperdataembed


Grab & merge the patent/paper info files


In [ ]:
potentialpairs = potentialpairs.merge(paperdataembed, on='work_id', how='left')
del paperdataembed
gc.collect()
potentialpairs.dropna(inplace=True)
potentialpairs 


In [ ]:
patentdataembed = pd.read_parquet(datadir + 'embeddedpatents.parquet')
patentdataembed = patentdataembed.rename(columns={'id': 'patent_id'})
patentdataembed['patent_id'] = patentdataembed['patent_id'].astype(str)
patentdataembed.drop_duplicates(subset=['patent_id'], inplace=True)
patentdataembed = patentdataembed.rename(columns={'encoded_title': 'encoded_patent_title'})  
patentdataembed = patentdataembed.rename(columns={'encoded_abstract': 'encoded_patent_abstract'})  

print(patentdataembed.shape)
# pppappdataembed


In [ ]:
potentialpairs = potentialpairs.merge(patentdataembed, on='patent_id', how='left')
del patentdataembed
gc.collect()
potentialpairs

merge the patent data and the paper data onto the patent/paper id pairs

In [ ]:
# print("Number of duplicate rows:", potentialpairs.duplicated().sum())
print("Missing values per column:\n", potentialpairs.isna().sum())
potentialpairs.dropna(inplace=True)
# potentialpairs.drop_duplicates(inplace=True)
potentialpairs


filter it by a window between patent and paper

merge in all the other characteristics, but do this after the window filter to save time/memory

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F

# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Configurable batch size (adjust based on VRAM and data size)
BATCH_SIZE = 1000000  # ~1.23GB for 4 tensors with D=768, adjust as needed

# Assume pppapppapersembed_window is the input DataFrame
# Get total number of samples
num_samples = len(potentialpairs)
print(f"Total samples: {num_samples}")

# Initialize columns for results
potentialpairs['title_similarity'] = np.nan
potentialpairs['abstract_similarity'] = np.nan

# Process data in batches
for start_idx in range(0, num_samples, BATCH_SIZE):
    end_idx = min(start_idx + BATCH_SIZE, num_samples)
    print(f"Processing batch: {start_idx} to {end_idx}")
    
    # Extract batch data
    batch_indices = range(start_idx, end_idx)
    batch_patent_title = torch.tensor(
        np.stack(potentialpairs['encoded_patent_title'].iloc[batch_indices].values),
        dtype=torch.float32
    ).to(device)
    batch_paper_title = torch.tensor(
        np.stack(potentialpairs['encoded_paper_title'].iloc[batch_indices].values),
        dtype=torch.float32
    ).to(device)
    batch_patent_abstract = torch.tensor(
        np.stack(potentialpairs['encoded_patent_abstract'].values[batch_indices]),
        dtype=torch.float32
    ).to(device)
    batch_paper_abstract = torch.tensor(
        np.stack(potentialpairs['encoded_paper_abstract'].values[batch_indices]),
        dtype=torch.float32
    ).to(device)
    
    # Compute cosine similarities
    title_similarities = F.cosine_similarity(batch_patent_title, batch_paper_title, dim=1)
    abstract_similarities = F.cosine_similarity(batch_patent_abstract, batch_paper_abstract, dim=1)
    
    # Compute norms and handle zero-norm cases
    title_norms = torch.norm(batch_patent_title, dim=1) * torch.norm(batch_paper_title, dim=1)
    abstract_norms = torch.norm(batch_patent_abstract, dim=1) * torch.norm(batch_paper_abstract, dim=1)
    title_similarities = torch.where(title_norms == 0, torch.tensor(float('nan'), device=device), title_similarities)
    abstract_similarities = torch.where(abstract_norms == 0, torch.tensor(float('nan'), device=device), abstract_similarities)
    
    # Store results in DataFrame
    potentialpairs.iloc[batch_indices, potentialpairs.columns.get_loc('title_similarity')] = (
        title_similarities.cpu().numpy()
    )
    potentialpairs.iloc[batch_indices, potentialpairs.columns.get_loc('abstract_similarity')] = (
        abstract_similarities.cpu().numpy()
    )
    
    # Clear GPU memory
    del batch_patent_title, batch_paper_title, batch_patent_abstract, batch_paper_abstract
    del title_similarities, abstract_similarities, title_norms, abstract_norms
    torch.cuda.empty_cache()

print("\nFinal DataFrame:")
print(potentialpairs[['work_id', 'patent_id', 'title_similarity', 'abstract_similarity']])

In [ ]:
columns_to_drop = ['encoded_patent_title', 'encoded_patent_abstract', 
                   'encoded_paper_title', 'encoded_paper_abstract']
potentialpairs = potentialpairs.drop(columns=columns_to_drop, errors='ignore')
potentialpairs
potentialpairs['notevenclose'] = (potentialpairs['title_similarity'] < 0.2) & (potentialpairs['abstract_similarity'] < 0.2)
potentialpairs['notevenclose'].value_counts()
potentialpairs = potentialpairs[potentialpairs['notevenclose'] == False]
potentialpairs = potentialpairs[['work_id', 'patent_id', 'title_similarity', 'abstract_similarity']]
potentialpairs

In [ ]:
potentialpairs.to_parquet(datadir + 'potentialpairs_with_similarity_scores.parquet', index=False, compression='snappy')
    # final_df.to_parquet(output_file, index=False, compression='snappy')


In [ ]:
potentialpairs = pd.read_parquet(datadir + 'potentialpairs_with_similarity_scores.parquet')
potentialpairs.shape

merge in the inventors and authors for the pairs that survived the temporal & similarity screens

In [ ]:
potentialpairs_patentsonly = potentialpairs['patent_id'].drop_duplicates().reset_index(drop=True)
potentialpairs_patentsonly
inventors = pd.read_csv(datadir + 'g_inventor_disambiguated.tsv.zip', sep='\t', compression='zip')
inventors = pd.merge(inventors, potentialpairs_patentsonly, on='patent_id', how='inner')
inventors = inventors[['patent_id', 'disambig_inventor_name_first', 'disambig_inventor_name_last']]
inventors.dropna(inplace=True)
inventors.drop_duplicates(inplace=True)
inventors['inventors'] = inventors['disambig_inventor_name_first'] + ' ' + inventors['disambig_inventor_name_last'] + '#'
# inventors=inventors[['patent_id', 'inventorsoneline']].drop_duplicates()
inventors['inventors'] = inventors.groupby('patent_id')['inventors'].transform(lambda x: ''.join(x))
inventors.drop_duplicates(inplace=True)
inventors['inventors'] = inventors['inventors'].str.replace(r'(.*)#$', r'\1', regex=True)
inventors=inventors[['patent_id', 'inventors']].drop_duplicates()
inventors['patent_id'] = inventors['patent_id'].astype(str)


In [ ]:
def format_number(x):
    if pd.isna(x):
        return x  # Preserve NaN
    if isinstance(x, float) and x.is_integer():
        return str(int(x))  # Convert to int, then to string
    return str(x)  # Keep as is for non-integer
potentialpairs_papersonly = potentialpairs['work_id'].drop_duplicates().reset_index(drop=True)
potentialpairs_papersonly
authors = pd.read_stata(datadir + "magauthorid.dta")
authors['magid'] = authors['magid'].apply(format_number)
authors['magid'] = authors['magid'].astype(str)
authors.dropna()
authors = pd.merge(authors, potentialpairs_papersonly, left_on='magid', right_on='work_id', how='inner')
authors
authors.dropna()
author_names = pd.read_stata(datadir + "authoridnames.dta")
authors = pd.merge(authors, author_names, on=['authorid'], how='left')
authors.drop_duplicates(inplace=True)
authors.dropna(inplace=True)
authors = authors[['work_id', 'authorname_normalized']]
authors.groupby('work_id').head(30).reset_index()
authors['authorname_normalized'] = authors.groupby('work_id')['authorname_normalized'].transform(lambda x: '#'.join(x))
authors = authors[['work_id', 'authorname_normalized']].drop_duplicates()
authors['authorname_normalized'] = authors['authorname_normalized'].str.replace(r'(.*)#$', r'\1', regex=True)
authors = authors.rename(columns={'authorname_normalized': 'authors'})
authors = authors[['work_id', 'authors']].drop_duplicates()
authors = authors.drop_duplicates().reset_index(drop=True)
authors

In [ ]:
potentialpairs = pd.merge(potentialpairs, inventors, on='patent_id', how='left')
potentialpairs = pd.merge(potentialpairs, authors, on='work_id', how='left')

In [ ]:
potentialpairs.drop_duplicates(inplace=True)
potentialpairs.dropna(inplace=True)
potentialpairs  

## Functions 

In [ ]:
## list the suffixes to remove and nobiliary particles to merge

name_del = set(["2nd", "3rd", "Jr", "Jr.", "Junior", "Sr", "Sr.", "Senior"])
ln_suff= set(["oster", "nordre", "vaster", "aust", "vesle", "da", "van t", "af", "al", "setya", "zu", "la", "na", "mic", "ofver", "el", "vetle", "van het", "dos", "ui", "vest", "ab", "vste", "nord", "van der", "bin", "ibn", "war", "fitz", "alam", "di", "erch", "fetch", "nga", "ka", "soder", "lille", "upp", "ua", "te", "ni", "bint", "von und zu", "vast", "vestre", "over", "syd", "mac", "nin", "nic", "putri", "bet", "verch", "norr", "bath", "della", "van", "ben", "du", "stor", "das", "neder", "abu", "degli", "vre", "ait", "ny", "opp", "pour", "kil", "der", "oz",  "von", "at", "nedre", "van den", "setia", "ap", "gil", "myljom", "van de", "stre", "dele", "mck", "de", "mellom", "mhic", "binti", "ath", "binte", "snder", "sre", "ned", "ter", "bar", "le", "mala", "ost", "syndre", "sr", "bat", "sndre", "austre", "putra", "putera", "av", "lu", "vetch", "ver", "puteri", "mc", "tre", "st"])


#suppress all the unwanted suffixes from a string. 
#name_del file can be modified if more or less suffixes want to be suppressed 


def name_delete(string):
    for elmt in name_del:
        if f" {elmt}" in string:
            string =  string.replace(f" {elmt}","")
    return string


#merge the nobiliary particles with the last name
#ln_suff file can be modified if more or less nobiliary particles want to be suppressed
    
def ln_suff_merge(string):
    for suff in ln_suff:
        if f"{' ' + suff + ' '}" in string or string.startswith(f"{suff + ' '}"):
            print(suff)
            string =  string.replace(f"{suff + ' '}", suff.replace(" ",""))
    return string



In [ ]:
#normalize a string dat that represents often a name. 

def normalize(data):
    normal = unicodedata.normalize('NFKD', data).encode('ASCII', 'ignore')
    val = normal.decode("utf-8")
    # delete unwanted elmt
    val = name_delete(val)
    # Lower case
    val = val.lower()
    # remove special characters
    val = re.sub('[^A-Za-z ]+', ' ', val)
    # remove multiple spaces
    val = re.sub(' +', ' ', val)
    # remove trailing spaces
    val = val.strip()
    return val

normalize("PIERRE De BEHRMAN")

In [ ]:
## calculate efficiently the dot product between two vectors

def norm(vector):
    return sqrt(sum(x * x for x in vector))    

def cosine_similarity2(vec_a, vec_b):
    norm_a = norm(vec_a)
    norm_b = norm(vec_b)
        
        
    if norm_a == 0 or norm_b == 0 :
        return None 
    
    else:
        dot = sum(a * b for a, b in zip(vec_a, vec_b))
        return dot / (norm_a * norm_b)

In [ ]:
#return a ratio of similarity of letters between two strings (to handle in the first names errors)
def match_ratio(string1,string2):
    return fuzz.ratio(string1, string2)

In [ ]:
#return 4 if string1 and string2 are the same
#return 3 if string1 and string2 sound the same
#otherwise, return less

def metaphone(string1,string2):
    if string1==string2:
        return 4
    tuple1 = doublemetaphone(string1)
    tuple2 = doublemetaphone(string2)
    if tuple1[0] == tuple2[0]:
        return 3
    elif tuple1[0] == tuple2[1] or tuple1[1] == tuple2[0]:
        return 2
    elif tuple1[1] == tuple2[1]:
        return 1
    else:
        return 0

In [ ]:
## return 1 if name1 and name2 potentially represent the same individual
## else return 0

def comparison(name1 , name2):
    
    #if there is no first name, retrun it's a match
    if name1 == "" or name2 == "":
        return 1
    
    #if some first names exist:
    list_name1 = name1.split()
    list_name2 = name2.split()
    
    #minimum number of first names to match
    number_match = min( len(list_name1) , len(list_name2))
    
    #for each name, check if there is a match
    count_match = 0
    for elem1 in list_name1:
        match = 0
        
        #if we just have the initial:
        if len(elem1) == 1:
            for elem2 in list_name2:
                if elem1[0] == elem2[0]:
                    match = 1
        else:
            for elem2 in list_name2:
                #if we just have the initial:
                if len(elem2) == 1 and elem1[0] == elem2[0]:
                    match = 1
                    
                #if elem1 and elem2 are entire first names that sound the same and have a ratio of common letters higher thsan 85%, it's a match
                elif len(elem2) > 1 and (metaphone(elem1,elem2) > 2 or match_ratio(elem1 , elem2) > 85 ) :
                    match = 1
                    
        #count the number of first names that match    
        count_match += match
        
    #check if we have enough first names that match 
    if count_match < number_match:
        return 0
    else:
        return 1



In [ ]:
## get number of in common authors

def number_of_in_common_authors(inventors , authors ):
    

    
    #count the number of names in common, and store the names in common
    count = 0
    list_in_common_authors = []

    
    for name_inventor in inventors:
        
        for name_author in authors:
                        
            
            if len(set(name_inventor.split()) & set( name_author.split())) > 0:
            
                match = comparison(name_author , name_inventor)

                #if the first names match, we store the first names that are matching and their index 
                if match == 1:
                    count += 1 
                    list_in_common_authors.append(name_author + "-" + name_inventor)
                    


    #return 1) the number of names in common, 2) the list of names in common, 3) their index 
    return  count ,  list_in_common_authors 


In [ ]:
import pandas as pd

# # Example DataFrame
# smallhope = pd.DataFrame({
#     'patent_id': ['P12345', 'P12346'],
#     'inventors': ['John Smith#Jane Doe', 'Bob Jones#Alice Brown'],
#     'authors': ['Jane Doe#Mike Lee', 'Bob Jones#Eve White']
# })

# # Example functions
# def normalize(name):
#     return name.strip().lower()

# def number_of_in_common_authors(inventors, authors):
#     common = len(set(inventors) & set(authors))
#     return (common, [])

# Vectorized processing
pppapppapersembed_window['inventors_list'] = pppapppapersembed_window['inventors'].apply(
    lambda x: [normalize(elem) for elem in set(x.split("#"))]
)
pppapppapersembed_window['authors_list'] = pppapppapersembed_window['authors'].apply(
    lambda x: [normalize(elem) for elem in set(x.split("#"))]
)

# Compute common authors
pppapppapersembed_window['authors in common'] = pppapppapersembed_window.apply(
    lambda row: number_of_in_common_authors(row['inventors_list'], row['authors_list'])[0], 
    axis=1
)

# Compute proportion
pppapppapersembed_window['proportion inventors'] = pppapppapersembed_window.apply(
    lambda row: 100 * (row['authors in common'] / len(row['inventors_list'])) 
    if row['inventors_list'] else 0, 
    axis=1
)

# Drop temporary columns
pppapppapersembed_window = pppapppapersembed_window.drop(['inventors_list', 'authors_list'], axis=1)

# Print results
print("\nFinal DataFrame:")
print(pppapppapersembed_window)

In [ ]:
## import predictive model trained using self-plagiarism
import pickle
with open( 'data/random_forest_PPPs.pkl','rb') as f:
    rf = pickle.load(f)
    

In [ ]:
# add empty field for existing RF
# pppapppapersembed_window['distance_inst_assignee'] = 0


In [ ]:
# rf.feature_names_in_



In [ ]:
pppapppapersembed_window['output'] = 0
send = pppapppapersembed_window[['title_similarity', 'abstract_similarity', 'authors in common',
       'proportion inventors']]

In [ ]:
print("Columns in pppapppapersembed_window:")
print(pppapppapersembed_window.columns.tolist())

In [ ]:
pppapppapersembed_window["predictions"] = rf.predict_proba(send)[:,1]


In [ ]:
pppapppapersembed_window
pppapppapersembed_window['predictions'].value_counts()

In [ ]:
pppapppapersembed_window["PPP_score"] = [ 4 if elem > 0.99 else 3 if elem > 0.9 else 2 if elem > 0.8 else 1 if elem > 0.7 else 0  for elem in pppapppapersembed_window["predictions"] ] 
# pppapppapersembed_window['PPP_score'] = 3 if (pppapppapersembed_window['PPP_score'] == 4) and ((pppapppapersembed_window['daysdiff'].dt.days) > 365) else pppapppapersembed_window['PPP_score']
condition = (pppapppapersembed_window['PPP_score'] == 4) & (pppapppapersembed_window['daysdiff'].dt.days > 365)
pppapppapersembed_window['PPP_score'] = np.where(condition, 3, pppapppapersembed_window['PPP_score'])
pppapppapersembed_window["PPP_score"].value_counts()

In [ ]:
processedreally = pppapppapersembed_window  
processedreally['goodinv'] = ((processedreally['authors in common']>1) & (processedreally['proportion inventors'] > 20)) | ((processedreally['authors in common']==1) & (processedreally['proportion inventors']>49))
processedreally['goodtitle'] = processedreally['title_similarity'] > .75    
processedreally['goodabstract'] = processedreally['abstract_similarity'] > .65
# processedreally = processedreally[processedreally['goodinv'] = True]
mask = (processedreally['goodinv'] == True) & ((processedreally['goodtitle'] == True) | (processedreally['goodabstract'] == True))
finalset = processedreally[mask].copy()
finalset = finalset[['patent_id', 'work_id']]
finalset.to_csv('data/pppappsNORF.csv', index=False)
finalset.shape

In [ ]:
ppps = pppapppapersembed_window[['patent_id', 'work_id', 'PPP_score']]
ppps = ppps[ppps['PPP_score'] > 0]  
ppps.to_csv('data/pppappsRF.csv', index=False)